# eXtreme Gradient Boosting Regression-pipelines (G4)

### Importación de librerias

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
import pickle
from xgboost import XGBRegressor

### Importación de Dataset

In [2]:
#Establecemos la ruta de la carpeta de trabajo y seleccionamos el archivo del dataset
path_base = r"C:\Users\ABUDABI\Downloads\CURSOS_ENEI_2026\ML_PRODUCCION_WEB\TRABAJO FINAL\REGRESION"
dataset = pd.read_csv(f'{path_base}/datos_ppto.csv', sep = ',')

#Inspeccionamos los 5 primeros registros del dataset
dataset.head(5)

,ano_eje,programa_pptal,tipo_prod_proy,departamento_meta,fuente_financ,categoria_gasto,mto_pia,mto_pim,mto_devenga_01,mto_devenga_02,mto_devenga_03,mto_devenga_04,mto_devenga_05,mto_devenga_06,mto_devenga_07,mto_devenga_08,mto_devenga_09,mto_devenga_10,mto_devenga_11,mto_devenga_12
0,2017,0093.DESARROLLO PRODUCTIVO DE LAS EMPRESAS,2.PROYECTO,07.PROVINCIA CONSTITUCIONAL DEL CALLAO,1.RECURSOS ORDINARIOS,6.GASTOS DE CAPITAL,6979843,0,0.0,0.0,0.0,0.00,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0
1,2017,0093.DESARROLLO PRODUCTIVO DE LAS EMPRESAS,2.PROYECTO,15.LIMA,1.RECURSOS ORDINARIOS,6.GASTOS DE CAPITAL,0,83164,0.0,0.0,0.0,83163.29,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0
2,2017,0093.DESARROLLO PRODUCTIVO DE LAS EMPRESAS,2.PROYECTO,15.LIMA,1.RECURSOS ORDINARIOS,6.GASTOS DE CAPITAL,0,0,0.0,0.0,0.0,0.00,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0
3,2017,0093.DESARROLLO PRODUCTIVO DE LAS EMPRESAS,2.PROYECTO,15.LIMA,1.RECURSOS ORDINARIOS,6.GASTOS DE CAPITAL,0,3492,0.0,0.0,0.0,0.00,0.0,3491.97,0.0,0.0,0.0,0.0,0.0,0.0
4,2017,0093.DESARROLLO PRODUCTIVO DE LAS EMPRESAS,2.PROYECTO,15.LIMA,1.RECURSOS ORDINARIOS,6.GASTOS DE CAPITAL,0,0,0.0,0.0,0.0,0.00,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0


### Agrupáción de las variables cuantitativas

In [3]:
#Agrupamos por las variables categoricas
# Definir variables categóricas (claves de agregación)
group_cols = [
    'ano_eje','programa_pptal','tipo_prod_proy','departamento_meta',
    'fuente_financ','categoria_gasto'
]

# Columnas numéricas a sumar
sum_cols = [
    'mto_pia',
    'mto_pim',
    'mto_devenga_01',
    'mto_devenga_02',
    'mto_devenga_03',
    'mto_devenga_04',
    'mto_devenga_05',
    'mto_devenga_06',
    'mto_devenga_07',
    'mto_devenga_08',
    'mto_devenga_09',
    'mto_devenga_10',
    'mto_devenga_11',
    'mto_devenga_12'
]

# Agregar dataset
dataset= (
    dataset
    .groupby(group_cols)[sum_cols]
    .sum()
    .reset_index()
)

### Agrupación de valores de las variables cualitativas

In [4]:
#Agrupamos los valores de la variable departamento_meta
frecuencias = dataset['departamento_meta'].value_counts()

categorias_validas = frecuencias[frecuencias > 10].index

dataset['departamento_meta'] = dataset['departamento_meta'].apply(
    lambda x: x if x in categorias_validas else 'OTROS_DEPARTAMENTOS'
)

### Generación de nuevas variables cuantitativas analíticas

In [5]:
#Generamos los devengados acumulados a cada mes (sumar devengados hasta el mes)
dataset['acum_01'] = dataset['mto_devenga_01']

for i in range(2,13):

    mes_actual = f"mto_devenga_{i:02d}"
    mes_anterior = f"acum_{i-1:02d}"
    mes_acum = f"acum_{i:02d}"

    dataset[mes_acum] = dataset[mes_anterior] + dataset[mes_actual]

#Generamos los porcentajes de devengado acumulado a cada mes (acumulado/PIM)
for i in range(1,13):

    col_acum = f"acum_{i:02d}"
    col_pct = f"pct_acum_{i:02d}"

    dataset[col_pct] = dataset[col_acum] / dataset['mto_pim']

#Generamos las variaciones de porcentajes de devengados acumulados en cada mes
for i in range(2, 13):

    mes_actual = f"pct_acum_{i:02d}"
    mes_anterior = f"pct_acum_{i-1:02d}"

    col_var = f"var_pct_{i:02d}"

    dataset[col_var] = dataset[mes_actual] - dataset[mes_anterior]

#Generamos el ratio de PIA sobre PIM
dataset['ratio_pia_pim'] = dataset['mto_pia'] / dataset['mto_pim']

In [6]:
#Seleccionamos las variables cuantitativas
cat_cols = [
    'ano_eje','programa_pptal','tipo_prod_proy','departamento_meta',
    'fuente_financ','categoria_gasto'
    ]

#Seleccionamos las variables cualitativas
num_cols = [
    'pct_acum_01','pct_acum_02','pct_acum_03','pct_acum_04','pct_acum_05',
    'pct_acum_06','pct_acum_07','pct_acum_08','pct_acum_09','pct_acum_10',
    'pct_acum_11','pct_acum_12','var_pct_02','var_pct_03','var_pct_04',
    'var_pct_05','var_pct_06','var_pct_07','var_pct_08','var_pct_09',
    'var_pct_10','var_pct_11','var_pct_12','ratio_pia_pim'
    ]

#Actualizamos con variables seleccionadas
dataset = dataset[cat_cols + num_cols]

In [7]:
#Verificamos el dataset actualizado
dataset.head(5)

,ano_eje,programa_pptal,tipo_prod_proy,departamento_meta,fuente_financ,categoria_gasto,pct_acum_01,pct_acum_02,pct_acum_03,pct_acum_04,...,var_pct_04,var_pct_05,var_pct_06,var_pct_07,var_pct_08,var_pct_09,var_pct_10,var_pct_11,var_pct_12,ratio_pia_pim
0,2017,0093.DESARROLLO PRODUCTIVO DE LAS EMPRESAS,2.PROYECTO,04.AREQUIPA,1.RECURSOS ORDINARIOS,6.GASTOS DE CAPITAL,0.005532,0.004508,0.010598,0.071295,...,0.060696,0.102689,0.104346,0.112750,0.109154,0.095017,0.042947,0.259124,0.085353,4.542888
1,2017,0093.DESARROLLO PRODUCTIVO DE LAS EMPRESAS,2.PROYECTO,07.PROVINCIA CONSTITUCIONAL DEL CALLAO,1.RECURSOS ORDINARIOS,6.GASTOS DE CAPITAL,0.000659,0.001082,0.003166,0.039278,...,0.036112,0.019815,0.000618,0.009734,0.003265,0.007826,0.007586,0.000000,0.813329,1.829056
2,2017,0093.DESARROLLO PRODUCTIVO DE LAS EMPRESAS,2.PROYECTO,10.HUANUCO,1.RECURSOS ORDINARIOS,6.GASTOS DE CAPITAL,0.000128,0.004022,0.005719,0.020567,...,0.014848,0.028980,0.011891,0.029177,0.002311,0.046282,0.088223,0.055371,0.113058,0.444622
3,2017,0093.DESARROLLO PRODUCTIVO DE LAS EMPRESAS,2.PROYECTO,11.ICA,1.RECURSOS ORDINARIOS,6.GASTOS DE CAPITAL,0.000000,0.135390,0.307023,0.443516,...,0.136494,0.102005,0.014245,-0.000846,0.020231,0.000000,0.030370,0.156564,0.067696,0.000000
4,2017,0093.DESARROLLO PRODUCTIVO DE LAS EMPRESAS,2.PROYECTO,12.JUNIN,1.RECURSOS ORDINARIOS,6.GASTOS DE CAPITAL,0.000000,0.006929,0.006929,0.010463,...,0.003534,0.003176,0.019293,0.000243,0.000000,0.000000,0.000983,-0.000289,0.000630,0.394116


In [8]:
#Eliminamos los valores NaN
dataset = dataset.dropna()

In [9]:
#Eliminamos aquellos registros cuyos valores sean menores a 0 y mayores a 1
#Seleccionamos las columnas numéricas de interés (pct_acum y var_pct)
columnas_numericas = dataset.select_dtypes(include=['float64']).columns

#Filtramos el dataset para mantener solo filas con valores entre 0 y 1
dataset = dataset[(dataset[columnas_numericas] >= 0).all(axis=1) &
                  (dataset[columnas_numericas] <= 1).all(axis=1)]

In [10]:
#Analizamos nuevamente la coherencia de los datos
dataset.describe()

,ano_eje,pct_acum_01,pct_acum_02,pct_acum_03,pct_acum_04,pct_acum_05,pct_acum_06,pct_acum_07,pct_acum_08,pct_acum_09,...,var_pct_04,var_pct_05,var_pct_06,var_pct_07,var_pct_08,var_pct_09,var_pct_10,var_pct_11,var_pct_12,ratio_pia_pim
count,682.000000,682.000000,682.000000,682.000000,682.000000,682.000000,682.000000,682.000000,682.000000,682.000000,...,682.000000,682.000000,682.000000,682.000000,682.000000,682.000000,682.000000,682.000000,682.000000,682.000000
mean,2021.431085,0.010942,0.037898,0.073872,0.115486,0.175936,0.224929,0.278726,0.340865,0.428007,...,0.041615,0.060450,0.048993,0.053797,0.062138,0.087142,0.063035,0.084230,0.254993,0.219793
std,2.334464,0.023107,0.097185,0.146608,0.191444,0.240662,0.275681,0.302492,0.327332,0.354031,...,0.102106,0.133558,0.116849,0.109481,0.122431,0.181115,0.120829,0.143049,0.303127,0.364312
min,2017.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2020.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.005436,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.021454,0.000000
50%,2022.000000,0.000000,0.000000,0.000000,0.000000,0.034013,0.093237,0.172178,0.269284,0.437558,...,0.000000,0.000000,0.000000,0.000000,0.004100,0.012082,0.012645,0.045055,0.131765,0.000000
75%,2023.000000,0.000000,0.040108,0.121699,0.216504,0.335483,0.424798,0.517470,0.615956,0.718105,...,0.062363,0.079178,0.069971,0.081600,0.081315,0.087752,0.082329,0.098968,0.371325,0.451188
max,2025.000000,0.116383,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,0.999996,1.000000,1.000000,0.999847,1.000000,1.000000,1.000000


### Determinación X e Y

In [11]:
#Definimos como y a pct_acum_09 y para X variables 3 meses antes PRESELECCIONADAS
X = dataset[['pct_acum_08','var_pct_08',
             'departamento_meta','categoria_gasto','ano_eje']]
y = dataset['pct_acum_09']

### Determinamos Train, Test y Val

In [12]:
#Por ser un planteamiento para un modelo temporal se trata de predecir con datos
#de años anteriores un valor para un año futuro

#Determinamos el TRAIN con información de 2017 a 2022
X_train = X[dataset['ano_eje'].between(2017, 2022)]
X_train = X_train.drop('ano_eje', axis=1)
y_train = y[dataset['ano_eje'].between(2017, 2022)]

#Determinamos el TEST con información de 2023 a 2024
X_test = X[dataset['ano_eje'].between(2023, 2024)]
X_test = X_test.drop('ano_eje', axis=1)
y_test = y[dataset['ano_eje'].between(2023, 2024)]

#Determinamos el VAL con información 2025
X_val = X[dataset['ano_eje'] == 2025]
X_val = X_val.drop('ano_eje', axis=1)
y_val = y[dataset['ano_eje'] == 2025]

### Creación del Pipeline

In [13]:
#Definimos columnas con variables cuantitativas y cualitativas
numeric_features = [
    'pct_acum_08','var_pct_08',
    ]

categorical_features = [
    'departamento_meta','categoria_gasto'
    ]

#Creamos el preprocesador
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ])

## Modelo eXtreme Gradient Boosting

### Pipeline

In [14]:
#Creamos el pipeline con el modelo XGBRegressor
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', XGBRegressor(
        objective='reg:squarederror',
        random_state=42
    ))
])

### Tunning

In [15]:
#Definimos hiperparámetros para ajustar
param_grid = {
    'regressor__n_estimators': [200, 400],
    'regressor__max_depth': [3, 5],
    'regressor__learning_rate': [0.05, 0.1],
    'regressor__subsample': [0.8, 1],
    'regressor__colsample_bytree': [0.8, 1]
}

#Creamos el objeto GridSearchCV
xgb_tunned = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
)

#Ajustamosel modelo al conjunto de entrenamiento
xgb_tunned.fit(X_train, y_train)

#Obtenemos las mejores configuraciones
best_params = xgb_tunned.best_params_

#Imprimimos las mejores configuraciones
print("Mejores hiperparámetros encontrados:", best_params)

Mejores hiperparámetros encontrados: {'regressor__colsample_bytree': 1, 'regressor__learning_rate': 0.05, 'regressor__max_depth': 3, 'regressor__n_estimators': 200, 'regressor__subsample': 0.8}


### Métricas

In [16]:
#Aplicamos el gb tuneado a Test, Train y Val

y_pred_test = xgb_tunned.predict(X_test)
y_pred_train = xgb_tunned.predict(X_train)
y_pred_val = xgb_tunned.predict(X_val)

rmse_XGB_test = mean_squared_error(y_test, y_pred_test)**0.5
rmse_XGB_train = mean_squared_error(y_train, y_pred_train)**0.5
rmse_XGB_val = mean_squared_error(y_val, y_pred_val)**0.5

# Imprimir métricas de evaluación
print("RMSE Test eXtreme Gradient Boosting:", rmse_XGB_test)
print("RMSE Train eXtreme Gradient Boosting:", rmse_XGB_train)
print("RMSE Val eXtreme Gradient Boosting:", rmse_XGB_val)

RMSE Test eXtreme Gradient Boosting: 0.17748871151371834
RMSE Train eXtreme Gradient Boosting: 0.13837725025334605
RMSE Val eXtreme Gradient Boosting: 0.16386438496075742


### Exportación de Modelo XGB

In [19]:
#Importamos la libreria
import os, sys
from joblib import dump

#Guardamos el modelo ajustado
dump(xgb_tunned, r'C:\Users\ABUDABI\Downloads\CURSOS_ENEI_2026\ML_PRODUCCION_WEB\TRABAJO FINAL\REGRESION\extremegradientboosting_model_reg.joblib')

['C:\\Users\\ABUDABI\\Downloads\\CURSOS_ENEI_2026\\ML_PRODUCCION_WEB\\TRABAJO FINAL\\REGRESION\\extremegradientboosting_model_reg.joblib']